In [39]:
import pandas as pd
import numpy as np

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,ID,Text_Oniric,Holograma_Matrix,Eticheta_Contextuala_Simbolica,Interconectare_Simbolica_Target
0,MNE_93810,ιφσηηφθσδδφιιθιλθιθηησιθλκιδσλλθσηηθθκφη,"[0.14967141530112327, 0.08617356988288154, 0.1...",Void,The empty abyssal region is seen as it dissolv...
1,MNE_24592,ξσδψδχθξφχξσσφφχηξφξθξσψσηψξφδχηφθφψψσχδξφθχφφ...,"[0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, ...",Grid,Observation: The rigid lattice framework colla...
2,MNE_13278,οσοθρσδρδοηησρωωοηρθοσωθδδοσηηηθδροφοθροθηθοδο...,"[0.09291631520839098, 0.10911248672533769, 0.1...",Void,The empty abyssal region stabilizes the surrou...
3,MNE_46048,οηφωσδωρηδοοηθσδ,"[0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, ...",Grid,Observation: The rigid lattice framework stabi...
4,MNE_42098,αασσσθηθαθφα,"[0.026393478145935026, 0.0, 0.1306619786981922...",Cluster,The dense node cluster expands rapidly into th...


In [46]:
def extract_mean(matrix): 
    matrix = matrix.replace("[", "").replace("]", "")
    numbers = matrix.split(", ")
    matrix = np.array(numbers).reshape(64, 64).astype("float64")
    meanie = matrix.mean()
    return meanie

extract_mean(df_train.Holograma_Matrix[4]) # daca incerci exemple se vede corelatia (0-2, 1-3 etc.)

0.11384815882866745

In [82]:
df_train['Matrix_Mean'] = df_train.Holograma_Matrix.apply(extract_mean)
df_test['Matrix_Mean'] = df_test.Holograma_Matrix.apply(extract_mean)

In [83]:
df_train['Text_Length'] = df_train.Text_Oniric.str.len()
df_test['Text_Length'] = df_test.Text_Oniric.str.len()

df_train.head()

,ID,Text_Oniric,Holograma_Matrix,Eticheta_Contextuala_Simbolica,Interconectare_Simbolica_Target,Matrix_Mean,Text_Length
0,MNE_93810,ιφσηηφθσδδφιιθιλθιθηησιθλκιδσλλθσηηθθκφη,"[0.14967141530112327, 0.08617356988288154, 0.1...",Void,The empty abyssal region is seen as it dissolv...,0.094070,40
1,MNE_24592,ξσδψδχθξφχξσσφφχηξφξθξσψσηψξφδχηφθφψψσχδξφθχφφ...,"[0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, ...",Grid,Observation: The rigid lattice framework colla...,0.174982,64
2,MNE_13278,οσοθρσδρδοηησρωωοηρθοσωθδδοσηηηθδροφοθροθηθοδο...,"[0.09291631520839098, 0.10911248672533769, 0.1...",Void,The empty abyssal region stabilizes the surrou...,0.093083,51
3,MNE_46048,οηφωσδωρηδοοηθσδ,"[0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, 0.4, ...",Grid,Observation: The rigid lattice framework stabi...,0.178146,16
4,MNE_42098,αασσσθηθαθφα,"[0.026393478145935026, 0.0, 0.1306619786981922...",Cluster,The dense node cluster expands rapidly into th...,0.113848,12


In [84]:
X = df_train["Text_Oniric"]
y1 = df_train["Eticheta_Contextuala_Simbolica"]
y2 = df_train["Interconectare_Simbolica_Target"]

X_final = df_test["Text_Oniric"]

from sklearn.feature_extraction.text import TfidfVectorizer

vect = TfidfVectorizer(analyzer='char', ngram_range=((2, 9))) # nu exista spatii deci facem pe caractere
X = vect.fit_transform(X)
X_final = vect.transform(X_final)

X

<5000x1162696 sparse matrix of type '<class 'numpy.float64'>'
	with 2044853 stored elements in Compressed Sparse Row format>

In [88]:
features = ["Matrix_Mean", "Text_Length"]

X_num = df_train[features]
X_num_final = df_test[features]

In [91]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score

model1 = LGBMClassifier(verbose=-1)
model2 = LogisticRegression()

print(f"Subtask 1 score - {cross_val_score(model1, X_num, y1, cv=5, scoring='f1_weighted').mean()}")
#print(f"Subtask 2 score - {cross_val_score(model2, X_num, y2, cv=5, scoring='accuracy').mean()}")

Subtask 1 score - 0.9951953481999883
Subtask 2 score - 0.034


In [92]:
model1.fit(X_num, y1)
preds_1 = model1.predict(X_num_final)

In [98]:
#model2.fit(X, y2)
#preds_2 = model2.predict(X_final)
preds_2 = ["The rigid lattice framework is seen as it dissolves into pure information, generating a faint emotional residue."] * len(df_test)

In [99]:
df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": df_test.ID,
    "answer": preds_1
})

df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.ID,
    "answer": preds_2
})

submission = pd.concat([df_1, df_2])

submission

,subtaskID,datapointID,answer
0,1,MNE_14965,Cluster
1,1,MNE_21471,Grid
2,1,MNE_19600,Wave
3,1,MNE_33329,Cluster
4,1,MNE_43018,Void
...,...,...,...
995,2,MNE_30319,The rigid lattice framework is seen as it diss...
996,2,MNE_68901,The rigid lattice framework is seen as it diss...
997,2,MNE_48754,The rigid lattice framework is seen as it diss...
998,2,MNE_80989,The rigid lattice framework is seen as it diss...


In [100]:
submission.to_csv("submission.csv", index=False)

In [102]:
df_train.Interconectare_Simbolica_Target.value_counts()

Interconectare_Simbolica_Target
The rigid lattice framework is seen as it dissolves into pure information, generating a faint emotional residue.           27
The dense node cluster stabilizes the surrounding field, thereby leaving no trace in the archives.                         27
Observation: The oscillating sine wave dissolves into pure information. Result: leaving no trace in the archives.          26
The oscillating sine wave is seen as it dissolves into pure information, generating a faint emotional residue.             25
The empty abyssal region is seen as it vibrates with unstable frequency, generating a faint emotional residue.             25
                                                                                                                           ..
The oscillating sine wave is seen as it vibrates with unstable frequency, generating a faint emotional residue.             9
The oscillating sine wave collapses under its own weight, thereby generating a faint e